In [1]:
# Inspect variable long names for QSOIL, Rh, TSOI files
import xarray as xr

files = {
    'QSOIL': 'trace.01-36.22000BP.clm2.QSOIL.22000BP_decavg_400BCE.nc',
    'Rh': 'trace.01-36.22000BP.clm2.Rh.22000BP_decavg_400BCE.nc',
    'TSOI': 'trace.01-36.22000BP.clm2.TSOI.22000BP_decavg_400BCE.nc',
}

for var_name, filename in files.items():
    print(f"\n{'='*60}")
    print(f"FILE: {filename}")
    print(f"{'='*60}")
    
    ds = xr.open_dataset(filename)
    
    # Find the main variable (same name as key or similar)
    for var in ds.data_vars:
        attrs = ds[var].attrs
        print(f"\nVariable: {var}")
        print(f"  Shape: {ds[var].shape}")
        print(f"  Dims: {ds[var].dims}")
        if 'long_name' in attrs:
            print(f"  Long name: {attrs['long_name']}")
        if 'units' in attrs:
            print(f"  Units: {attrs['units']}")
    
    ds.close()


FILE: trace.01-36.22000BP.clm2.QSOIL.22000BP_decavg_400BCE.nc

Variable: BSW
  Shape: (10, 48, 96)
  Dims: ('levsoi', 'lat', 'lon')
  Long name: slope of soil water retention curve
  Units: unitless

Variable: DZSOI
  Shape: (10, 48, 96)
  Dims: ('levsoi', 'lat', 'lon')
  Long name: soil thickness
  Units: m

Variable: QSOIL
  Shape: (2204, 48, 96)
  Dims: ('time', 'lat', 'lon')
  Long name: ground evaporation
  Units: mm/s

Variable: SUCSAT
  Shape: (10, 48, 96)
  Dims: ('levsoi', 'lat', 'lon')
  Long name: saturated soil matric potential
  Units: mm

Variable: WATSAT
  Shape: (10, 48, 96)
  Dims: ('levsoi', 'lat', 'lon')
  Long name: saturated soil water content (porosity)
  Units: mm3/mm3

Variable: ZSOI
  Shape: (10, 48, 96)
  Dims: ('levsoi', 'lat', 'lon')
  Long name: soil depth
  Units: m

Variable: area
  Shape: (48, 96)
  Dims: ('lat', 'lon')
  Long name: grid cell areas
  Units: km^2

Variable: date_written
  Shape: (2204,)
  Dims: ('time',)

Variable: landfrac
  Shape: (48,

In [4]:
# Check if Rh is all zeros and plot on global map
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

ds_rh = xr.open_dataset('trace.01-36.22000BP.clm2.Rh.22000BP_decavg_400BCE.nc')
rh = ds_rh.Rh

print(f"Rh shape: {rh.shape}")
print(f"Rh dims: {rh.dims}")
print(f"\nRh statistics:")
print(f"  Min: {float(np.nanmin(rh.values)):.6e}")
print(f"  Max: {float(np.nanmax(rh.values)):.6e}")
print(f"  Mean: {float(np.nanmean(rh.values)):.6e}")

# Check for NaNs
n_nan = np.isnan(rh.values).sum()
n_zero = (rh.values == 0).sum()
print(f"  NaN count: {n_nan}")
print(f"  Zero count: {n_zero}")
print(f"  Total: {rh.size}")

if float(np.nanmax(rh.values)) == 0:
    print("\n⚠️ Rh file contains ALL ZEROS - no heterotrophic respiration data!")
else:
    # Plot time-mean on global map (sum over PFTs first if 4D)
    if rh.ndim == 4:
        rh_2d = rh.sum(dim='pft').mean(dim='time')
    else:
        rh_2d = rh.mean(dim='time')
    
    fig = plt.figure(figsize=(12, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.Robinson())
    
    lon = ds_rh.lon.values
    lat = ds_rh.lat.values
    lon_plot = np.where(lon > 180, lon - 360, lon)
    sort_idx = np.argsort(lon_plot)
    lon_sorted = lon_plot[sort_idx]
    rh_sorted = rh_2d.values[:, sort_idx]
    
    lon_mesh, lat_mesh = np.meshgrid(lon_sorted, lat)
    
    im = ax.pcolormesh(lon_mesh, lat_mesh, rh_sorted, 
                       transform=ccrs.PlateCarree(), 
                       cmap='YlOrRd', shading='auto')
    ax.coastlines(linewidth=0.5)
    ax.set_global()
    ax.set_title(f'Rh - Time Mean\n{rh.attrs.get("long_name", "")} [{rh.attrs.get("units", "")}]')
    plt.colorbar(im, ax=ax, orientation='horizontal', pad=0.05, shrink=0.8)
    
    plt.tight_layout()
    plt.show()

ds_rh.close()

Rh shape: (2204, 10, 48, 96)
Rh dims: ('time', 'pft', 'lat', 'lon')

Rh statistics:
  Min: 0.000000e+00
  Max: 0.000000e+00
  Mean: 0.000000e+00
  NaN count: 56382320
  Zero count: 45178000
  Total: 101560320

⚠️ Rh file contains ALL ZEROS - no heterotrophic respiration data!


In [ ]:
# Check each PFT separately to see if any have data
ds_rh = xr.open_dataset('trace.01-36.22000BP.clm2.Rh.22000BP_decavg_400BCE.nc')
rh = ds_rh.Rh

print(f"Rh shape: {rh.shape}")
print(f"Rh dims: {rh.dims}")

# Check PFT dimension
if 'pft' in rh.dims:
    print(f"\nPFT values: {ds_rh.pft.values if 'pft' in ds_rh.coords else 'not in coords'}")
    
    print("\n=== Statistics by PFT ===")
    for i in range(rh.shape[1]):  # pft is dimension 1
        pft_data = rh.isel(pft=i).values
        pft_min = np.nanmin(pft_data)
        pft_max = np.nanmax(pft_data)
        pft_mean = np.nanmean(pft_data)
        n_nonzero = np.sum(pft_data != 0)
        n_nan = np.isnan(pft_data).sum()
        print(f"  PFT {i}: min={pft_min:.2e}, max={pft_max:.2e}, mean={pft_mean:.2e}, non-zero={n_nonzero}, NaN={n_nan}")

# Also check if summing over PFTs gives anything
rh_sum_pft = rh.sum(dim='pft')
print(f"\nSum over all PFTs:")
print(f"  Min: {float(np.nanmin(rh_sum_pft.values)):.2e}")
print(f"  Max: {float(np.nanmax(rh_sum_pft.values)):.2e}")
print(f"  Mean: {float(np.nanmean(rh_sum_pft.values)):.2e}")

ds_rh.close()